# Tratando os dados das estações de qualidade do ar

Este notebook consolida os dados de **qualidade do ar** provenientes das **estações fixas** e das **unidades móveis** que serão utilizados na etapa de análise exploratória.

O foco é: leitura dos dados brutos, **padronização de nomes e colunas**, integração das bases e geração de uma tabela única pronta para as próximas etapas.

> **Escopo**: unir medições e metadados das estações, harmonizar diferenças de nomenclatura entre as fontes e salvar um arquivo consolidado em formato `.csv`.


## Objetivos desta etapa
1. **Carregar** os dados brutos de estações fixas, medições de qualidade do ar e unidades móveis.
2. **Padronizar** nomes de colunas e identificadores para permitir a integração entre as bases.
3. **Consolidar** todas as observações em uma única tabela e **exportar** o resultado para uso posterior.


## Configurações e importações
Bibliotecas usadas e parâmetros gerais do notebook.


In [1]:
import pandas as pd
from pathlib import Path

## Carregamento dos dados

Nesta etapa, carregamos diretamente do GitHub os arquivos brutos necessários para a consolidação:
- cadastro/metadados das estações;
- medições das estações fixas;
- medições das unidades móveis.


In [2]:
url_sensors = "https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/refs/heads/Refactoring-And-Documentation/Data/RawData/MonitorAr/dados_sensores.parquet"
url_qualiar_sensors = "https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/refs/heads/Refactoring-And-Documentation/Data/RawData/MonitorAr/dados_qualidade_ar_sensores.parquet"
url_qualiar_mobile_units = "https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/refs/heads/Refactoring-And-Documentation/Data/RawData/MonitorAr/dados_qualidade_ar_unidades_moveis.parquet"

df_sensors = pd.read_parquet(url_sensors)
df_qualia_sensors = pd.read_parquet(url_qualiar_sensors)
df_qualia_mobile_units = pd.read_parquet(url_qualiar_mobile_units)


## Renomeando e padronizando colunas

As fontes não usam exatamente a mesma convenção de nomes. Aqui ajustamos os campos
necessários para que as tabelas possam ser combinadas sem ambiguidade.


In [9]:
df_sensors.rename(columns={'CodNum': 'codnum'}, inplace=True)
df_qualia_mobile_units.rename(columns={'estacao': 'Nome', 'long': 'lon'}, inplace=True)


## Unificando informações de qualidade do ar

Primeiro, associamos as medições das estações fixas aos metadados por meio de `codnum`.
Em seguida, concatenamos as observações das unidades móveis para formar uma base única.


In [11]:
df_merged = pd.merge(df_qualia_sensors, df_sensors, on='codnum', how='left')

df_merged = pd.concat([df_merged, df_qualia_mobile_units], ignore_index=True)

df_merged.drop(columns=['codnum', 'estação'], inplace=True, errors='ignore')

df_merged.sort_values('data', inplace=True)

df_merged


,data,chuva,temp,ur,so2,no2,co,no,nox,o3,pm10,pm2_5,lat,lon,Nome
0,2012-01-01 00:30:00,0.2,24.67,95.24,NaN,15.18,0.42,2.18,17.36,28.06,81.0,NaN,-22.887910,-43.471074,ESTAÇÃO BANGU
245472,2012-01-01 00:30:00,0.8,24.07,99.56,NaN,NaN,NaN,NaN,NaN,24.52,74.0,NaN,-23.004379,-43.629010,ESTAÇÃO PEDRA DE GUARATIBA
122736,2012-01-01 00:30:00,0.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-22.908344,-43.178152,ESTAÇÃO CENTRO
306839,2012-01-01 00:30:00,0.2,27.14,99.82,8.41,NaN,0.17,NaN,NaN,10.95,49.0,NaN,-22.897771,-43.221745,ESTAÇÃO SÃO CRISTÓVÃO
368207,2012-01-01 00:30:00,0.0,21.47,98.40,1.51,NaN,0.06,NaN,NaN,24.55,36.0,NaN,-22.924915,-43.232657,ESTAÇÃO TIJUCA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
490942,2018-12-31 23:30:00,0.0,NaN,NaN,NaN,NaN,0.01,NaN,NaN,11.87,53.0,NaN,-22.965004,-43.180482,ESTAÇÃO COPACABANA
184103,2018-12-31 23:30:00,0.0,27.23,83.68,NaN,NaN,0.66,NaN,NaN,24.68,29.0,NaN,-22.908344,-43.178152,ESTAÇÃO CENTRO
61367,2018-12-31 23:30:00,0.0,26.45,99.13,1.74,40.71,1.28,3.28,44.00,3.82,80.0,NaN,-22.887910,-43.471074,ESTAÇÃO BANGU
368206,2018-12-31 23:30:00,0.0,27.23,NaN,NaN,NaN,1.02,NaN,NaN,4.35,49.0,NaN,-22.897771,-43.221745,ESTAÇÃO SÃO CRISTÓVÃO


## Padronizando os nomes das unidades móveis

As unidades móveis aparecem com siglas curtas em parte dos dados. Nesta etapa, essas siglas
são substituídas por nomes mais descritivos, facilitando a leitura e a interpretação da base final.


In [ ]:
df_merged['Nome'] = df_merged['Nome'].replace({
    'UMRC': 'UNIDADE MOVEL RECREIO',
    'UMGM': 'UNIDADE MOVEL BANGU',
    
    'UMCP': 'UNIDADE MOVEL COPACABANA',
    'UMMD': 'UNIDADE MOVEL MADUREIRA',
    
    'UMDC': 'UNIDADE MOVEL DELCASTILHO',
    'UMMA': 'UNIDADE MOVEL MARACANA',
    
    'UMPR': 'UNIDADE MOVEL GAMBOA',
    'UMCA': 'UNIDADE MOVEL CAJU',
    'UMCJ': 'UNIDADE MOVEL CAJU',
    'UMCAJU': 'UNIDADE MOVEL CAJU',
    'UMPM': 'UNIDADE MOVEL CENTRO'
})

df_merged.reset_index(drop=True, inplace=True)

## Ordenando as colunas

Reorganizamos a tabela final para deixar as colunas mais importantes — como `data` e `Nome` —
no início da visualização, melhorando a legibilidade da base consolidada.


In [14]:
colunas_prioritarias = [col for col in ['data', 'Nome'] if col in df_merged.columns]
outras_colunas = [col for col in df_merged.columns if col not in colunas_prioritarias]

df_merged = df_merged[colunas_prioritarias + outras_colunas]

df_merged

,data,Nome,chuva,temp,ur,so2,no2,co,no,nox,o3,pm10,pm2_5,lat,lon
0,2012-01-01 00:30:00,ESTAÇÃO BANGU,0.2,24.67,95.24,NaN,15.18,0.42,2.18,17.36,28.06,81.0,NaN,-22.887910,-43.471074
1,2012-01-01 00:30:00,ESTAÇÃO PEDRA DE GUARATIBA,0.8,24.07,99.56,NaN,NaN,NaN,NaN,NaN,24.52,74.0,NaN,-23.004379,-43.629010
2,2012-01-01 00:30:00,ESTAÇÃO CENTRO,0.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-22.908344,-43.178152
3,2012-01-01 00:30:00,ESTAÇÃO SÃO CRISTÓVÃO,0.2,27.14,99.82,8.41,NaN,0.17,NaN,NaN,10.95,49.0,NaN,-22.897771,-43.221745
4,2012-01-01 00:30:00,ESTAÇÃO TIJUCA,0.0,21.47,98.40,1.51,NaN,0.06,NaN,NaN,24.55,36.0,NaN,-22.924915,-43.232657
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
532687,2018-12-31 23:30:00,ESTAÇÃO COPACABANA,0.0,NaN,NaN,NaN,NaN,0.01,NaN,NaN,11.87,53.0,NaN,-22.965004,-43.180482
532688,2018-12-31 23:30:00,ESTAÇÃO CENTRO,0.0,27.23,83.68,NaN,NaN,0.66,NaN,NaN,24.68,29.0,NaN,-22.908344,-43.178152
532689,2018-12-31 23:30:00,ESTAÇÃO BANGU,0.0,26.45,99.13,1.74,40.71,1.28,3.28,44.00,3.82,80.0,NaN,-22.887910,-43.471074
532690,2018-12-31 23:30:00,ESTAÇÃO SÃO CRISTÓVÃO,0.0,27.23,NaN,NaN,NaN,1.02,NaN,NaN,4.35,49.0,NaN,-22.897771,-43.221745


## Gerando CSV de saída

Por fim, salvamos a base consolidada em formato `.csv` dentro da pasta de dados intermediários do projeto.


In [15]:
project_root = Path().resolve().parents[2]
output_dir = project_root / "Data" / "IntermediaryData" / "MonitorAr" / "InitialMergedData"
output_dir.mkdir(parents=True, exist_ok=True)

output_csv_path = output_dir / "merged_air_quality_data.csv"
df_merged.to_csv(output_csv_path, index=False, encoding="utf-8")

print(f"Arquivo salvo em: {output_csv_path}")


Arquivo salvo em: C:\Users\João Henrique\OneDrive - cefet-rj.br\Projetos-2026\qualiar\Data\IntermediaryData\MonitorAr\InitialMergedData\merged_air_quality_data.csv
